In [ ]:
"""
Final_Analysis.py  --  the single script that produces EVERY number, table and
figure in the thesis Results chapter.  Run it in Colab (or locally).

PRINCIPLE
    No result in the thesis is typed in by hand or calculated outside this
    script.  The Results chapter is written FROM the files this script saves
    to the folder  final_results/  and can cite them as
    "calculated with Final_Analysis.py (see Appendix)".

INPUT (all in BASE_DIR)
    all_participants_trial_summary.csv      (made by Combine_and_Analyze.py)
    participant_<id>_godspeed_scores.csv    (item-level Godspeed answers;
                                             used only for Cronbach's alpha)

OUTPUT (all in BASE_DIR/final_results/)
    00_run_log.txt                       everything printed + versions + input file hash
    01_data_completeness.csv             which trials/participants entered the analysis
    02_descriptives_by_condition.csv     n, mean, SD, median per measure and condition
    03_friedman_tests.csv                condition effects (chi2, df, p, Kendall's W, Holm p)
    04_correlations_pooled.csv           the 55 measure x Godspeed correlations
                                           (Pearson + Spearman, 95% CI, Holm p)
    05_correlations_robustness.csv       the same 55 pairs, three ways of checking they are
                                           not artefacts of pooling 24 trials from 5 people:
                                           within-participant, within-condition,
                                           leave-one-participant-out
    06_godspeed_by_condition.csv         Godspeed ratings per condition + Friedman tests
    07_godspeed_reliability.csv          Cronbach's alpha per Godspeed subscale
    08_sensitivity_excluding_trials.csv  the 11 + 55 tests again without the trials in SENSITIVITY_EXCLUDE
    10_heart_vs_camera_correlations.csv  RMSSD delta vs each camera measure (10 correlations)
    11_sensors_vs_godspeed_side_by_side.csv  each measure's r with the other sensor and with Godspeed
    09_fps_correction_effect.csv         how much each trial mean changed after the frame-rate correction
    fig1_au_means_by_condition.png       fig2_arousal_vs_safety.png
    fig3_rmssd_delta_by_condition.png    README_outputs.txt  (what each file is)

WHAT IS PRIMARY vs EXPLORATORY (fixed before looking at results; see Methodology)
    primary     : RMSSD delta, AU4, AU5, AU7, AU9, arousal, valence
    exploratory : AU1, AU2, AU20 (fear-associated), AU17 (disgust-associated)
    -> 11 condition tests and 55 correlation tests in total.  The p-values in the
       main columns are UNCORRECTED (as stated in the Limitations); the columns
       called p_holm show the same tests after Holm correction, as a transparency
       check, not as the headline result.
"""

import datetime
import hashlib
import io
import os
import platform
import sys

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/ThesisExperiment_cleann"
OUT_DIR_NAME = "final_results"
PARTICIPANT_IDS = [1, 3, 4, 5, 6]
CONDITIONS = ["Robot1", "Robot2", "Robot3", "Robot4", "Human"]  # presentation order
SUMMARY_FILE = "all_participants_trial_summary.csv"

PRIMARY = {
    "rmssd_delta": "RMSSD delta (ms)",
    "AU04_mean": "AU4", "AU05_mean": "AU5", "AU07_mean": "AU7",
    "AU09_mean": "AU9", "arousal_mean": "Arousal", "valence_mean": "Valence",
}
EXPLORATORY = {
    "AU01_mean": "AU1", "AU02_mean": "AU2", "AU20_mean": "AU20",
    "AU17_mean": "AU17",
}
TIER = {**{k: "primary" for k in PRIMARY}, **{k: "exploratory" for k in EXPLORATORY}}
LABEL = {**PRIMARY, **EXPLORATORY}
MEASURES = list(PRIMARY) + list(EXPLORATORY)

GODSPEED = ["anthropomorphism", "animacy", "likeability",
            "perceived_intelligence", "perceived_safety"]
GODSPEED_ITEMS = {
    "anthropomorphism": [f"anthropomorphism_{i}" for i in range(1, 6)],
    "animacy": [f"animacy_{i}" for i in range(1, 7)],
    "likeability": [f"likeability_{i}" for i in range(1, 6)],
    "perceived_intelligence": [f"intelligence_{i}" for i in range(1, 6)],
    "perceived_safety": [f"safety_{i}" for i in range(1, 4)],
}
ALPHA = 0.05

# Trials left out in the sensitivity check (file 08): (participant, condition).
# P5 Human ran at 26.5 fps in the capture-rate check.
SENSITIVITY_EXCLUDE = [(5, "Human")]
# Copy of the summary file made BEFORE the frame-rate correction (file 09).
OLD_SUMMARY_FILE = "all_participants_trial_summary_BEFORE_fps_correction.csv"


# ----------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------
class Tee(io.TextIOBase):
    """Send print output to the screen AND keep a copy for the log file."""
    def __init__(self, stream):
        self.stream, self.buf = stream, io.StringIO()
    def write(self, s):
        self.buf.write(s)
        return self.stream.write(s)
    def flush(self):
        self.stream.flush()


def holm(pvals):
    """Holm-Bonferroni adjusted p-values (same order as input)."""
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(1.0, running)
    return adj


def fisher_ci(r, n, level=0.95):
    if n <= 3 or abs(r) >= 1:
        return (np.nan, np.nan)
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    crit = stats.norm.ppf(0.5 + level / 2)
    return (np.tanh(z - crit * se), np.tanh(z + crit * se))


def pearson_with_df(x, y, df):
    """Pearson r, with the p-value computed for a given number of degrees of freedom."""
    if len(x) < 4 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan
    r = np.corrcoef(x, y)[0, 1]
    if df < 1 or abs(r) >= 1:
        return r, np.nan
    t = r * np.sqrt(df / (1 - r ** 2))
    return r, 2 * stats.t.sf(abs(t), df)


def cronbach_alpha(items):
    items = items.dropna()
    k = items.shape[1]
    if k < 2 or len(items) < 3:
        return np.nan
    item_var = items.var(axis=0, ddof=1).sum()
    total_var = items.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_var / total_var) if total_var > 0 else np.nan


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def sensor_sensor_tables(df, df_s, co, path):
    """Heart vs camera. Correlates RMSSD delta (heart) with every camera measure
    (AUs, arousal, valence) across the analysed trials, and puts those results
    next to the Godspeed correlations of the same measures (files 10 and 11)."""
    heart = "rmssd_delta"
    cams = [m for m in MEASURES if m != heart]
    rows = []
    for c in cams:
        pair = df[["participant_id", "robot_name", heart, c]].dropna().copy()
        n = len(pair)
        r, p = stats.pearsonr(pair[heart], pair[c])
        rho, p_rho = stats.spearmanr(pair[heart], pair[c])
        lo, hi = fisher_ci(r, n)
        k_part, k_cond = pair["participant_id"].nunique(), pair["robot_name"].nunique()
        xa = pair[heart] - pair.groupby("participant_id")[heart].transform("mean")
        ya = pair[c] - pair.groupby("participant_id")[c].transform("mean")
        r_wp, p_wp = pearson_with_df(xa.values, ya.values, n - k_part - 1)
        xb = pair[heart] - pair.groupby("robot_name")[heart].transform("mean")
        yb = pair[c] - pair.groupby("robot_name")[c].transform("mean")
        r_wc, p_wc = pearson_with_df(xb.values, yb.values, n - k_cond - 1)
        loo = []
        for pid in pair["participant_id"].unique():
            sub = pair[pair["participant_id"] != pid]
            if len(sub) > 3 and sub[heart].std() > 0 and sub[c].std() > 0:
                loo.append(stats.pearsonr(sub[heart], sub[c])[0])
        ps = df_s[[heart, c]].dropna()
        r_s, p_s = stats.pearsonr(ps[heart], ps[c])
        rows.append({"heart_measure": heart, "camera_measure": c, "label": LABEL[c],
                     "tier": TIER[c], "N": n, "pearson_r": r, "pearson_p": p,
                     "ci95_low": lo, "ci95_high": hi,
                     "spearman_rho": rho, "spearman_p": p_rho,
                     "r_within_participant": r_wp, "p_within_participant": p_wp,
                     "r_within_condition": r_wc, "p_within_condition": p_wc,
                     "loo_r_min": min(loo) if loo else np.nan,
                     "loo_r_max": max(loo) if loo else np.nan,
                     "loo_sign_stays_same": bool(loo) and all(np.sign(x) == np.sign(r) for x in loo),
                     "sens_N": len(ps), "sens_r": r_s, "sens_p": p_s})
    ss = pd.DataFrame(rows)
    ss["pearson_p_holm_all_%d" % len(ss)] = holm(ss["pearson_p"])
    ss["significant_uncorrected"] = ss["pearson_p"] < ALPHA
    ss["significant_after_holm"] = ss["pearson_p_holm_all_%d" % len(ss)] < ALPHA
    ss = ss.sort_values("pearson_p").reset_index(drop=True)
    ss.to_csv(path("10_heart_vs_camera_correlations.csv"), index=False)
    print("\n=== 10 Heart (RMSSD delta) vs camera measures: %d tests; %d with p<.05 uncorrected, %d after Holm ===" %
          (len(ss), int(ss["significant_uncorrected"].sum()), int(ss["significant_after_holm"].sum())))
    print(ss[["camera_measure", "N", "pearson_r", "pearson_p", "ci95_low", "ci95_high",
              "spearman_rho", "r_within_participant", "p_within_participant",
              "pearson_p_holm_all_%d" % len(ss)]].round(4).to_string(index=False))

    # side-by-side: Pearson r of each measure with the Godspeed subscales (from file 04)
    r_tab = co.pivot(index="measure", columns="godspeed_subscale", values="pearson_r")
    p_tab = co.pivot(index="measure", columns="godspeed_subscale", values="pearson_p")
    comp = []
    for m in MEASURES:
        row = {"measure": m, "label": LABEL[m], "tier": TIER[m]}
        if m == heart:
            row["r_with_rmssd_delta"] = np.nan
            row["p_with_rmssd_delta"] = np.nan
        else:
            hit = ss[ss["camera_measure"] == m].iloc[0]
            row["r_with_rmssd_delta"] = hit["pearson_r"]
            row["p_with_rmssd_delta"] = hit["pearson_p"]
        for g in GODSPEED:
            row["r_" + g] = r_tab.loc[m, g]
            row["p_" + g] = p_tab.loc[m, g]
        comp.append(row)
    comp = pd.DataFrame(comp)
    comp.to_csv(path("11_sensors_vs_godspeed_side_by_side.csv"), index=False)
    print("\n=== 11 Each measure: r with the other sensor (RMSSD delta) and r with each Godspeed subscale ===")
    print(comp[["measure", "r_with_rmssd_delta"] + ["r_" + g for g in GODSPEED]].round(3).to_string(index=False))
    return ss, comp


def main(base_dir=BASE_DIR):
    tee = Tee(sys.stdout)
    sys.stdout = tee
    try:
        _run(base_dir)
    finally:
        sys.stdout = tee.stream
        out = os.path.join(base_dir, OUT_DIR_NAME, "00_run_log.txt")
        with open(out, "w", encoding="utf8") as f:
            f.write(tee.buf.getvalue())
        print(f"\nRun log saved: {out}")


def _run(base_dir):
    out_dir = os.path.join(base_dir, OUT_DIR_NAME)
    os.makedirs(out_dir, exist_ok=True)
    path = lambda name: os.path.join(out_dir, name)

    summary_path = os.path.join(base_dir, SUMMARY_FILE)
    print("Final_Analysis.py")
    print(f"Run at (UTC): {datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%S}")
    print(f"Python {platform.python_version()} | pandas {pd.__version__} | "
          f"numpy {np.__version__} | scipy {__import__('scipy').__version__} | "
          f"matplotlib {matplotlib.__version__}")
    print(f"Input: {summary_path}")
    print(f"Input SHA-256: {sha256(summary_path)}")

    raw = pd.read_csv(summary_path)
    raw = raw[raw["participant_id"].isin(PARTICIPANT_IDS)].copy()

    # ---------------- 01 data completeness ----------------
    if "excluded_cross_participant" in raw.columns:
        raw["excluded_cross_participant"] = raw["excluded_cross_participant"].astype(str).str.lower().eq("true")
    else:
        raw["excluded_cross_participant"] = False
    df = raw[~raw["excluded_cross_participant"]].copy()

    comp = (raw.assign(included=~raw["excluded_cross_participant"])
               .pivot_table(index="participant_id", columns="robot_name",
                            values="included", aggfunc="first")
               .reindex(columns=CONDITIONS))
    comp.to_csv(path("01_data_completeness.csv"))
    n_part = raw["participant_id"].nunique()
    complete_ids = sorted(df.groupby("participant_id")["robot_name"].nunique()
                          .loc[lambda s: s == len(CONDITIONS)].index)
    print(f"\nParticipants in file: {n_part} | trials in file: {len(raw)} | "
          f"trials analysed: {len(df)} | excluded trials: {int(raw['excluded_cross_participant'].sum())}")
    print(f"Participants with all 5 conditions (used in Friedman tests): {complete_ids}")
    missing = [m for m in MEASURES + GODSPEED if m not in df.columns]
    if missing:
        raise SystemExit(f"Missing columns in summary file: {missing}")
    print("Missing values per analysed measure:")
    print(df[MEASURES + GODSPEED].isna().sum().to_string())

    # ---------------- 02 descriptives ----------------
    rows = []
    for m in MEASURES:
        for c in CONDITIONS:
            v = df.loc[df["robot_name"] == c, m].dropna()
            rows.append({"measure": m, "label": LABEL[m], "tier": TIER[m],
                         "condition": c, "n": len(v), "mean": v.mean(),
                         "sd": v.std(ddof=1), "median": v.median(),
                         "min": v.min(), "max": v.max()})
    desc = pd.DataFrame(rows)
    desc.to_csv(path("02_descriptives_by_condition.csv"), index=False)
    print("\n=== 02 Descriptives (mean) by condition, presentation order ===")
    print(desc.pivot(index="measure", columns="condition", values="mean")
              .reindex(index=MEASURES, columns=CONDITIONS).round(3).to_string())

    # ---------------- 03 Friedman tests ----------------
    def friedman(measure, frame):
        piv = (frame.pivot_table(index="participant_id", columns="robot_name", values=measure)
                    .reindex(columns=CONDITIONS).dropna())
        if len(piv) < 2:
            return None
        chi2, p = stats.friedmanchisquare(*[piv[c] for c in CONDITIONS])
        k = len(CONDITIONS)
        return {"N": len(piv), "chi2": chi2, "df": k - 1, "p": p,
                "kendalls_W": chi2 / (len(piv) * (k - 1))}

    frows = []
    for m in MEASURES:
        res = friedman(m, df)
        frows.append({"measure": m, "label": LABEL[m], "tier": TIER[m], **res})
    fr = pd.DataFrame(frows)
    fr["p_holm_all_11"] = holm(fr["p"])
    fr["significant_uncorrected"] = fr["p"] < ALPHA
    fr["significant_after_holm"] = fr["p_holm_all_11"] < ALPHA
    fr.to_csv(path("03_friedman_tests.csv"), index=False)
    print("\n=== 03 Friedman tests across the 5 conditions ===")
    print(fr[["measure", "tier", "N", "chi2", "p", "kendalls_W", "p_holm_all_11"]]
          .round(4).to_string(index=False))

    # ---------------- 04 pooled correlations ----------------
    crow = []
    for m in MEASURES:
        for g in GODSPEED:
            pair = df[["participant_id", "robot_name", m, g]].dropna()
            n = len(pair)
            r, p = stats.pearsonr(pair[m], pair[g])
            rho, p_rho = stats.spearmanr(pair[m], pair[g])
            lo, hi = fisher_ci(r, n)
            crow.append({"measure": m, "label": LABEL[m], "tier": TIER[m],
                         "godspeed_subscale": g, "N": n, "pearson_r": r,
                         "pearson_p": p, "ci95_low": lo, "ci95_high": hi,
                         "spearman_rho": rho, "spearman_p": p_rho})
    co = pd.DataFrame(crow)
    co["pearson_p_holm_all_55"] = holm(co["pearson_p"])
    co["significant_uncorrected"] = co["pearson_p"] < ALPHA
    co["significant_after_holm"] = co["pearson_p_holm_all_55"] < ALPHA
    co = co.sort_values("pearson_p").reset_index(drop=True)
    co.to_csv(path("04_correlations_pooled.csv"), index=False)
    print(f"\n=== 04 Pooled correlations: {len(co)} tests; "
          f"{int(co['significant_uncorrected'].sum())} with p<.05 uncorrected, "
          f"{int(co['significant_after_holm'].sum())} after Holm ===")
    print(co[["measure", "godspeed_subscale", "N", "pearson_r", "pearson_p",
              "ci95_low", "ci95_high", "spearman_rho", "pearson_p_holm_all_55"]]
          .head(10).round(4).to_string(index=False))

    # ---------------- 05 robustness of the correlations ----------------
    rob = []
    for m in MEASURES:
        for g in GODSPEED:
            pair = df[["participant_id", "robot_name", m, g]].dropna().copy()
            n = len(pair)
            k_part = pair["participant_id"].nunique()
            k_cond = pair["robot_name"].nunique()
            # (a) within-participant: remove each person's own mean from both variables.
            xa = pair[m] - pair.groupby("participant_id")[m].transform("mean")
            ya = pair[g] - pair.groupby("participant_id")[g].transform("mean")
            r_wp, p_wp = pearson_with_df(xa.values, ya.values, n - k_part - 1)
            # (b) within-condition: remove each condition's mean from both variables.
            xb = pair[m] - pair.groupby("robot_name")[m].transform("mean")
            yb = pair[g] - pair.groupby("robot_name")[g].transform("mean")
            r_wc, p_wc = pearson_with_df(xb.values, yb.values, n - k_cond - 1)
            # (c) leave one participant out.
            loo = []
            for pid in pair["participant_id"].unique():
                sub = pair[pair["participant_id"] != pid]
                if len(sub) > 3 and sub[m].std() > 0 and sub[g].std() > 0:
                    loo.append(stats.pearsonr(sub[m], sub[g])[0])
            r_full = stats.pearsonr(pair[m], pair[g])[0]
            rob.append({
                "measure": m, "label": LABEL[m], "tier": TIER[m],
                "godspeed_subscale": g, "N": n, "pearson_r_pooled": r_full,
                "r_within_participant": r_wp, "p_within_participant": p_wp,
                "r_within_condition": r_wc, "p_within_condition": p_wc,
                "loo_r_min": min(loo) if loo else np.nan,
                "loo_r_max": max(loo) if loo else np.nan,
                "loo_sign_stays_same": bool(loo) and all(np.sign(x) == np.sign(r_full) for x in loo),
            })
    rb = pd.DataFrame(rob).sort_values("p_within_participant").reset_index(drop=True)
    rb.to_csv(path("05_correlations_robustness.csv"), index=False)
    print("\n=== 05 Robustness of the strongest pooled correlations ===")
    top = co.head(6)[["measure", "godspeed_subscale"]]
    show = rb.merge(top, on=["measure", "godspeed_subscale"])
    print(show[["measure", "godspeed_subscale", "pearson_r_pooled",
                "r_within_participant", "p_within_participant",
                "r_within_condition", "p_within_condition",
                "loo_r_min", "loo_r_max", "loo_sign_stays_same"]].round(3).to_string(index=False))

    # ---------------- 06 Godspeed ratings by condition ----------------
    g_rows, g_fr = [], []
    for g in GODSPEED:
        for c in CONDITIONS:
            v = df.loc[df["robot_name"] == c, g].dropna()
            g_rows.append({"godspeed_subscale": g, "condition": c, "n": len(v),
                           "mean": v.mean(), "sd": v.std(ddof=1)})
        res = friedman(g, df)
        g_fr.append({"godspeed_subscale": g, **res})
    gd = pd.DataFrame(g_rows)
    gf = pd.DataFrame(g_fr)
    gf["p_holm_5"] = holm(gf["p"])
    pd.concat([gd.assign(table="descriptives"), gf.assign(table="friedman")],
              ignore_index=True).to_csv(path("06_godspeed_by_condition.csv"), index=False)
    print("\n=== 06 Godspeed ratings (mean) by condition ===")
    print(gd.pivot(index="godspeed_subscale", columns="condition", values="mean")
            .reindex(columns=CONDITIONS).round(2).to_string())
    print(gf[["godspeed_subscale", "N", "chi2", "p", "kendalls_W", "p_holm_5"]].round(4).to_string(index=False))

    # ---------------- 07 reliability (Cronbach's alpha) ----------------
    item_frames = []
    for pid in PARTICIPANT_IDS:
        f = os.path.join(base_dir, f"participant_{pid}_godspeed_scores.csv")
        if os.path.exists(f):
            t = pd.read_csv(f)
            t["participant_id"] = pid
            item_frames.append(t)
        else:
            print(f"  [!] {f} not found; alpha computed without participant {pid}.")
    arows = []
    if item_frames:
        items = pd.concat(item_frames, ignore_index=True)
        excl = raw.loc[raw["excluded_cross_participant"], ["participant_id", "robot_name"]]
        for _, e in excl.iterrows():
            items = items[~((items["participant_id"] == e["participant_id"]) &
                            (items["robot_name"] == e["robot_name"]))]
        for g, cols in GODSPEED_ITEMS.items():
            cols = [c for c in cols if c in items.columns]
            arows.append({"godspeed_subscale": g, "n_items": len(cols),
                          "n_trials": int(items[cols].dropna().shape[0]) if cols else 0,
                          "cronbach_alpha": cronbach_alpha(items[cols]) if cols else np.nan})
    al = pd.DataFrame(arows)
    al.to_csv(path("07_godspeed_reliability.csv"), index=False)
    print("\n=== 07 Cronbach's alpha (all analysed trials pooled) ===")
    print(al.round(3).to_string(index=False) if len(al) else "no item-level files found")

    # ---------------- 08 sensitivity: leave out named trials ----------------
    # Repeats the 11 condition tests and 55 correlations without the trials in
    # SENSITIVITY_EXCLUDE, to show whether any result depends on them.
    keep = ~df.apply(lambda r: (int(r["participant_id"]), r["robot_name"]) in SENSITIVITY_EXCLUDE, axis=1)
    df_s = df[keep].copy()
    srows = []
    for m in MEASURES:
        a, b = friedman(m, df), friedman(m, df_s)
        srows.append({"test": "friedman", "measure": m, "godspeed_subscale": "",
                      "main_N": a["N"], "main_stat": a["chi2"], "main_p": a["p"],
                      "sens_N": b["N"] if b else np.nan,
                      "sens_stat": b["chi2"] if b else np.nan,
                      "sens_p": b["p"] if b else np.nan})
    for m in MEASURES:
        for g in GODSPEED:
            pa = df[[m, g]].dropna()
            pb = df_s[[m, g]].dropna()
            ra, qa = stats.pearsonr(pa[m], pa[g])
            rb_, qb = stats.pearsonr(pb[m], pb[g])
            srows.append({"test": "pearson", "measure": m, "godspeed_subscale": g,
                          "main_N": len(pa), "main_stat": ra, "main_p": qa,
                          "sens_N": len(pb), "sens_stat": rb_, "sens_p": qb})
    sens = pd.DataFrame(srows)
    sens["same_direction"] = np.sign(sens["main_stat"]) == np.sign(sens["sens_stat"])
    sens["significant_main"] = sens["main_p"] < ALPHA
    sens["significant_sens"] = sens["sens_p"] < ALPHA
    sens["conclusion_changes"] = sens["significant_main"] != sens["significant_sens"]
    sens.to_csv(path("08_sensitivity_excluding_trials.csv"), index=False)
    print(f"\n=== 08 Sensitivity: results without {SENSITIVITY_EXCLUDE} ===")
    print(f"Tests whose significance (p<.05) changes when those trials are left out: "
          f"{int(sens['conclusion_changes'].sum())} of {len(sens)}")
    print(sens[sens["significant_main"] | sens["significant_sens"]]
          [["test", "measure", "godspeed_subscale", "main_N", "main_stat", "main_p",
            "sens_N", "sens_stat", "sens_p"]].round(4).to_string(index=False))

    # ---------------- 09 effect of the frame-rate correction ----------------
    old_path = os.path.join(base_dir, OLD_SUMMARY_FILE)
    if os.path.exists(old_path):
        old = pd.read_csv(old_path)
        mrg = raw.merge(old, on=["participant_id", "robot_name"], suffixes=("_new", "_old"))
        crows = []
        for m in MEASURES:
            for _, r in mrg.iterrows():
                crows.append({"participant_id": r["participant_id"], "robot_name": r["robot_name"],
                              "measure": m, "before_correction": r[f"{m}_old"],
                              "after_correction": r[f"{m}_new"],
                              "change": r[f"{m}_new"] - r[f"{m}_old"]})
        ch = pd.DataFrame(crows)
        ch.to_csv(path("09_fps_correction_effect.csv"), index=False)
        sd_between = df.groupby("robot_name")[MEASURES].mean().std(ddof=1)
        summ = (ch.groupby("measure")["change"].agg(lambda x: x.abs().max()).rename("max_abs_change")
                .to_frame())
        summ["between_condition_SD_of_means"] = sd_between
        summ["max_change_pct_of_SD"] = 100 * summ["max_abs_change"] / summ["between_condition_SD_of_means"]
        print("\n=== 09 Effect of the frame-rate correction on trial means ===")
        print(summ.reindex(MEASURES).round(4).to_string())
        big = ch.loc[ch.assign(a=ch["change"].abs()).groupby("measure")["a"].idxmax()]
        print("\nTrial with the largest change per measure:")
        print(big[["measure", "participant_id", "robot_name", "before_correction",
                   "after_correction", "change"]].round(4).to_string(index=False))
    else:
        print(f"\n(09 skipped: {OLD_SUMMARY_FILE} not found. Keep a copy of the summary file "
              f"from BEFORE the frame-rate correction under that name to see its effect.)")

    # ---------------- figures ----------------
    cmap = plt.get_cmap("tab10")
    pid_color = {pid: cmap(i) for i, pid in enumerate(PARTICIPANT_IDS)}
    xs = np.arange(len(CONDITIONS))

    def bar_panel(ax, m):
        means = [df.loc[df["robot_name"] == c, m].mean() for c in CONDITIONS]
        sds = [df.loc[df["robot_name"] == c, m].std(ddof=1) for c in CONDITIONS]
        ns = [df.loc[df["robot_name"] == c, m].count() for c in CONDITIONS]
        sem = [s / np.sqrt(n) if n > 1 else 0 for s, n in zip(sds, ns)]
        ax.bar(xs, means, yerr=sem, capsize=3, color="#9bb7d4", edgecolor="#33516e")
        for pid in PARTICIPANT_IDS:
            sub = df[df["participant_id"] == pid].set_index("robot_name")[m].reindex(CONDITIONS)
            ax.plot(xs, sub.values, "o", ms=3.5, color=pid_color[pid], alpha=.85)
        ax.set_xticks(xs)
        ax.set_xticklabels(["R1", "R2", "R3", "R4", "Human"])
        ax.set_title(LABEL[m], fontsize=10)

    aus = ["AU04_mean", "AU05_mean", "AU07_mean", "AU09_mean", "AU01_mean"]
    fig, axes = plt.subplots(1, len(aus), figsize=(3 * len(aus), 3.2), sharex=True)
    for ax, m in zip(axes, aus):
        bar_panel(ax, m)
    axes[0].set_ylabel("Mean AU intensity (Detectorv2)")
    handles = [plt.Line2D([], [], marker="o", ls="", color=pid_color[p], label=f"P{p}") for p in PARTICIPANT_IDS]
    fig.legend(handles=handles, loc="lower center", ncol=len(PARTICIPANT_IDS), frameon=False)
    fig.suptitle("Facial Action Units by condition, in presentation order "
                 "(bars: mean ± SEM; dots: participants)", fontsize=10)
    fig.tight_layout(rect=[0, 0.07, 1, 0.94])
    fig.savefig(path("fig1_au_means_by_condition.png"), dpi=200)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    pair = df[["participant_id", "robot_name", "arousal_mean", "perceived_safety"]].dropna()
    for pid in PARTICIPANT_IDS:
        s = pair[pair["participant_id"] == pid]
        ax.scatter(s["arousal_mean"], s["perceived_safety"], color=pid_color[pid], label=f"P{pid}", s=34)
    slope, intercept = np.polyfit(pair["arousal_mean"], pair["perceived_safety"], 1)
    xx = np.linspace(pair["arousal_mean"].min(), pair["arousal_mean"].max(), 50)
    ax.plot(xx, slope * xx + intercept, color="black", lw=1)
    r, p = stats.pearsonr(pair["arousal_mean"], pair["perceived_safety"])
    ax.set_xlabel("Arousal (Detectorv2, trial mean)")
    ax.set_ylabel("Perceived Safety (Godspeed, 1-5)")
    ax.set_title(f"Arousal vs Perceived Safety (r = {r:.2f}, p = {p:.4f}, n = {len(pair)} trials)", fontsize=9)
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(path("fig2_arousal_vs_safety.png"), dpi=200)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(5.2, 3.8))
    bar_panel(ax, "rmssd_delta")
    ax.axhline(0, color="black", lw=.8)
    ax.set_ylabel("RMSSD delta: stimulus - baseline (ms)")
    ax.set_title("RMSSD change from baseline by condition (bars: mean ± SEM; dots: participants)", fontsize=8)
    fig.tight_layout()
    fig.savefig(path("fig3_rmssd_delta_by_condition.png"), dpi=200)
    plt.close(fig)

    # ---------------- 10/11 heart vs camera, and both vs Godspeed ----------------
    sensor_sensor_tables(df, df_s, co, path)

    # ---------------- README of outputs ----------------
    readme = (
        "Files produced by Final_Analysis.py\n\n"
        "00_run_log.txt                    everything printed by the script, library versions, input file hash\n"
        "01_data_completeness.csv          which participant x condition trials were included (True) or excluded (False)\n"
        "02_descriptives_by_condition.csv  n, mean, SD, median, min, max per measure and condition\n"
        "03_friedman_tests.csv             11 condition tests: chi2, df, p, Kendall's W, Holm-adjusted p\n"
        "04_correlations_pooled.csv        55 measure x Godspeed correlations (Pearson + Spearman, 95% CI, Holm p)\n"
        "05_correlations_robustness.csv    same 55 pairs: within-participant, within-condition, leave-one-participant-out\n"
        "06_godspeed_by_condition.csv      Godspeed ratings per condition + Friedman tests (5 subscales)\n"
        "07_godspeed_reliability.csv       Cronbach's alpha per Godspeed subscale\n"
        "08_sensitivity_excluding_trials.csv  same tests without the trials in SENSITIVITY_EXCLUDE\n"
        "09_fps_correction_effect.csv      change in each trial mean after the frame-rate correction\n"
        "10_heart_vs_camera_correlations.csv  RMSSD delta (heart) vs each camera measure: Pearson/Spearman, CI, within-participant, within-condition, leave-one-out, sensitivity, Holm\n"
        "11_sensors_vs_godspeed_side_by_side.csv  each measure: r with RMSSD delta and r with each Godspeed subscale\n"
        "fig1_au_means_by_condition.png    AU means by condition (presentation order)\n"
        "fig2_arousal_vs_safety.png        arousal vs Perceived Safety, one colour per participant\n"
        "fig3_rmssd_delta_by_condition.png RMSSD delta by condition\n"
    )
    with open(path("README_outputs.txt"), "w", encoding="utf8") as f:
        f.write(readme)
    print("\nAll outputs saved to:", out_dir)
    print(readme)


if __name__ == "__main__":
    if not os.path.isdir(BASE_DIR):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except ImportError:
            pass
    main(BASE_DIR)